In [1]:
import pandas as pd
import sys
sys.path.append('../Source')

from Pipeline.build_comparisons import buildComparisons

alignedData = pd.read_csv('../Data/Processed/alignedData.csv')
modelingData = buildComparisons(alignedData)

print(f'Shape: {modelingData.shape[0]} rows, {modelingData.shape[1]} columns')
modelingData.head()





Shape: 2346 rows, 5 columns


,week,regionI,regionJ,winsI,total
0,2018-01-01/2018-01-07,CaucasusCA,Europe,1050,1505
1,2018-01-01/2018-01-07,CaucasusCA,MiddleEast,1050,3660
2,2018-01-01/2018-01-07,CaucasusCA,NorthAfrica,1050,1320
3,2018-01-01/2018-01-07,Europe,MiddleEast,455,3065
4,2018-01-01/2018-01-07,Europe,NorthAfrica,455,725


In [2]:
import pandas as pd
import numpy as np
import arviz as az 

In [3]:
regions = ['CaucasusCA', 'Europe', 'MiddleEast', 'NorthAfrica']
regionMap = {'CaucasusCA' : 0, 'Europe' : 1, 'MiddleEast' : 2, 'NorthAfrica' : 3}

winsI = modelingData['winsI'].to_numpy()
totals = modelingData['total'].to_numpy()

indexI = modelingData['regionI'].map(regionMap).to_numpy()
indexJ = modelingData['regionJ'].map(regionMap).to_numpy()

nRegions = len(regions)

print(indexI[:10])
print(indexJ[:10])


[0 0 0 1 1 2 0 0 0 1]
[1 2 3 2 3 3 1 2 3 2]


In [4]:
import pymc as pm

with pm.Model() as model:
    # Establish Bayesian prior, constrained to sum to 0
    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nRegions)

    strengthI = beta[indexI]
    strengthJ = beta[indexJ]

    logOdds = strengthI - strengthJ
    probability = pm.math.sigmoid(logOdds)
     
    likelihood = pm.Binomial('likelihood', n=totals, p=probability, observed=winsI)

with model: 
    trace = pm.sample(1000, tune=1000, return_inferencedata=True)

Initializing NUTS using jitter+adapt_diag...
c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\pytensor\link\c\cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 20 seconds.


In [5]:
az.summary(trace, var_names=['beta'])

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta[0],-0.686,0.001,-0.688,-0.685,0.0,0.0,5624.0,3001.0,1.0
beta[1],0.946,0.001,0.945,0.947,0.0,0.0,4473.0,3545.0,1.0
beta[2],0.849,0.001,0.848,0.850,0.0,0.0,4464.0,3364.0,1.0
beta[3],-1.109,0.001,-1.110,-1.107,0.0,0.0,2809.0,2847.0,1.0


In [6]:
samples = trace.posterior['beta'].values.reshape(-1, nRegions)

print(samples.shape)
print(samples[:5])

(4000, 4)
[[-0.68725961  0.94670481  0.84921019 -1.10865539]
 [-0.68621237  0.94572502  0.84936846 -1.10888111]
 [-0.68567358  0.94657969  0.84869784 -1.10960395]
 [-0.68470871  0.94499121  0.8487885  -1.109071  ]
 [-0.6851195   0.94520017  0.84920185 -1.10928252]]


In [7]:
rankings = np.argsort(-samples, axis=1)
print(rankings[:10])

[[1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]]


In [8]:
uniqueRankings, counts = np.unique(rankings, axis=0, return_counts=True)
print(uniqueRankings)
print(counts)

[[1 2 0 3]]
[4000]


In [9]:
from scipy.stats import entropy

# Convert raw counts to proportions (that sum to 1)
probabilities = counts / counts.sum()
entropyValue = entropy(probabilities)

print(probabilities)
print(entropyValue)



[1.]
0.0


In [10]:
pairwiseFlip = (samples[:, 1] > samples[:, 2]).mean()
europeOverMiddleEast = pairwiseFlip

print(europeOverMiddleEast)

1.0


In [11]:
import itertools

flipProbabilities = dict()

for regionI, regionJ in itertools.combinations(range(nRegions), 2):
    probIOverJ = (samples[:, regionI] > samples[:, regionJ]).mean()
    pairLabel = f'{regions[regionI]} over {regions[regionJ]}'
    flipProbabilities[pairLabel] = probIOverJ

for pair, probability in flipProbabilities.items():
    print(pair,probability)



CaucasusCA over Europe 0.0
CaucasusCA over MiddleEast 0.0
CaucasusCA over NorthAfrica 1.0
Europe over MiddleEast 1.0
Europe over NorthAfrica 1.0
MiddleEast over NorthAfrica 1.0


In [12]:
hdiBounds = az.hdi(trace, var_names=['beta'])
print(hdiBounds)

<xarray.Dataset> Size: 144B
Dimensions:     (beta_dim_0: 4, hdi: 2)
Coordinates:
  * beta_dim_0  (beta_dim_0) int64 32B 0 1 2 3
  * hdi         (hdi) <U6 48B 'lower' 'higher'
Data variables:
    beta        (beta_dim_0, hdi) float64 64B -0.6877 -0.6848 ... -1.11 -1.107
Attributes:
    created_at:                 2026-07-07T22:33:36.658649+00:00
    arviz_version:              0.23.4
    inference_library:          pymc
    inference_library_version:  5.28.5
    sampling_time:              19.895038843154907
    tuning_steps:               1000


In [13]:
hdiArray = hdiBounds['beta'].values
print(hdiArray)

[[-0.68772914 -0.68484763]
 [ 0.94494765  0.94722451]
 [ 0.84784787  0.85009428]
 [-1.11044817 -1.10725409]]


In [14]:
def intervalOverlap(lowerI, upperI, lowerJ, upperJ):
    return (lowerI <= upperJ) and (lowerJ <= upperI)

In [15]:
hdiOverlaps = dict()

for regionI, regionJ in itertools.combinations(range(nRegions), 2):
    lowerI, upperI = hdiArray[regionI]
    lowerJ, upperJ = hdiArray[regionJ]
    overlaps = intervalOverlap(lowerI, upperI, lowerJ, upperJ)
    pairLabel = f'{regions[regionI]} vs {regions[regionJ]}'
    hdiOverlaps[pairLabel] = overlaps

for pair, overlaps in hdiOverlaps.items():
    print(pair, overlaps)


CaucasusCA vs Europe False
CaucasusCA vs MiddleEast False
CaucasusCA vs NorthAfrica False
Europe vs MiddleEast False
Europe vs NorthAfrica False
MiddleEast vs NorthAfrica False


In [16]:
from Pipeline.uncertainty_metrics import computeUncertainty

results = computeUncertainty(trace, regions)
print(results['entropy'])
print(results['flipProbabilities'])
print(results['hdiOverlaps'])

0.0
{'CaucasusCA over Europe': np.float64(0.0), 'CaucasusCA over MiddleEast': np.float64(0.0), 'CaucasusCA over NorthAfrica': np.float64(1.0), 'Europe over MiddleEast': np.float64(1.0), 'Europe over NorthAfrica': np.float64(1.0), 'MiddleEast over NorthAfrica': np.float64(1.0)}
{'CaucasusCA vs Europe': np.False_, 'CaucasusCA vs MiddleEast': np.False_, 'CaucasusCA vs NorthAfrica': np.False_, 'Europe vs MiddleEast': np.False_, 'Europe vs NorthAfrica': np.False_, 'MiddleEast vs NorthAfrica': np.False_}


In [17]:
import pandas as pd

acledRaw = pd.read_csv('../Data/Raw/ACLED Data_2026-06-24.csv')

uniqueCountries = acledRaw['country'].nunique()
print(uniqueCountries)

countryCounts = acledRaw['country'].value_counts()
print(countryCounts)

83
country
Ukraine                  244529
Syria                    111325
Yemen                     78809
Palestine                 76999
Afghanistan               55147
                          ...  
Bailiwick of Guernsey        13
Isle of Man                  12
Faroe Islands                 8
Vatican City                  8
Gibraltar                     4
Name: count, Length: 83, dtype: int64


In [18]:
regionValues = acledRaw['region'].unique()
print(regionValues)

<ArrowStringArray>
['Northern Africa', 'Middle East', 'Europe', 'Caucasus and Central Asia']
Length: 4, dtype: str


In [19]:
regionalACLED = acledRaw[acledRaw['region'].isin(['Northern Africa', 'Middle East', 'Europe', 'Caucasus and Central Asia'])]

uniqueCountriesRegional = regionalACLED['country'].nunique()
print(uniqueCountriesRegional)

countryCountsRegional = regionalACLED.groupby('region')['country'].nunique()
print(countryCountsRegional)

83
region
Caucasus and Central Asia     9
Europe                       53
Middle East                  15
Northern Africa               6
Name: country, dtype: int64


In [20]:
countryEventCounts = regionalACLED.groupby('country').size().sort_values(ascending=False)
print(countryEventCounts.describe())
print(countryEventCounts.tail(20))

count        83.000000
mean      13098.445783
std       32366.510652
min           4.000000
25%         333.000000
50%        2230.000000
75%        7857.500000
max      244529.000000
dtype: float64
country
Iceland                  264
Luxembourg               208
Tajikistan               175
Kuwait                   111
Qatar                     62
Turkmenistan              51
Oman                      48
Greenland                 48
United Arab Emirates      45
San Marino                33
Andorra                   29
Monaco                    19
Akrotiri and Dhekelia     15
Bailiwick of Jersey       15
Liechtenstein             15
Bailiwick of Guernsey     13
Isle of Man               12
Faroe Islands              8
Vatican City               8
Gibraltar                  4
dtype: int64


In [21]:
survivingCountries = countryEventCounts[countryEventCounts >= 50].index
regionalACLEDFiltered = regionalACLED[regionalACLED['country'].isin(survivingCountries)]

print(regionalACLEDFiltered.groupby('region')['country'].nunique())

region
Caucasus and Central Asia     9
Europe                       41
Middle East                  13
Northern Africa               6
Name: country, dtype: int64


In [22]:
marketReturns = pd.read_csv('../Data/Processed/marketDF.csv', index_col=0, parse_dates=True)

print(marketReturns.head())
print(marketReturns.index.dtype)



            Close_EEM   Close_GLD  Close_ITA  Close_USO  Close_VIX
Date                                                              
2018-01-02  39.776558  125.150002  86.677139  96.559998       9.77
2018-01-03  40.157669  124.820000  86.805916  98.720001       9.15
2018-01-04  40.356510  125.459999  87.413139  98.959999       9.22
2018-01-05  40.704487  125.330002  88.199722  98.480003       9.22
2018-01-08  40.704487  125.309998  88.733322  99.040001       9.52
datetime64[us]


In [23]:
returnColumns = ['Close_EEM', 'Close_GLD', 'Close_ITA', 'Close_USO', 'Close_VIX']

marketReturns[['Return_EEM', 'Return_GLD', 'Return_ITA', 'Return_USO', 'Return_VIX']] = marketReturns[returnColumns].pct_change()

print(marketReturns.head())

            Close_EEM   Close_GLD  Close_ITA  Close_USO  Close_VIX  \
Date                                                                 
2018-01-02  39.776558  125.150002  86.677139  96.559998       9.77   
2018-01-03  40.157669  124.820000  86.805916  98.720001       9.15   
2018-01-04  40.356510  125.459999  87.413139  98.959999       9.22   
2018-01-05  40.704487  125.330002  88.199722  98.480003       9.22   
2018-01-08  40.704487  125.309998  88.733322  99.040001       9.52   

            Return_EEM  Return_GLD  Return_ITA  Return_USO  Return_VIX  
Date                                                                    
2018-01-02         NaN         NaN         NaN         NaN         NaN  
2018-01-03    0.009581   -0.002637    0.001486    0.022370   -0.063460  
2018-01-04    0.004952    0.005127    0.006995    0.002431    0.007650  
2018-01-05    0.008623   -0.001036    0.008998   -0.004850    0.000000  
2018-01-08    0.000000   -0.000160    0.006050    0.005686    0.032538 

In [24]:
tradingDays = marketReturns.index
print(tradingDays[:5])

DatetimeIndex(['2018-01-02', '2018-01-03', '2018-01-04', '2018-01-05',
               '2018-01-08'],
              dtype='datetime64[us]', name='Date', freq=None)


In [25]:
from Pipeline.align_events import matchToTradingDay, getEventWindow

regionalACLEDFiltered['tradingDay'] = regionalACLEDFiltered['event_date'].apply(
    lambda eventDate: matchToTradingDay(eventDate, tradingDays)
)

print(regionalACLEDFiltered[['event_date', 'tradingDay']].head())

   event_date tradingDay
0  2018-01-01 2018-01-02
1  2018-01-01 2018-01-02
2  2018-01-01 2018-01-02
3  2018-01-01 2018-01-02
4  2018-01-01 2018-01-02


In [26]:
dailyCountryCounts = regionalACLEDFiltered.groupby(['tradingDay', 'country']).size().reset_index(name = 'eventCount')
print(dailyCountryCounts.head())

  tradingDay      country  eventCount
0 2018-01-02  Afghanistan          92
1 2018-01-02      Armenia          10
2 2018-01-02   Azerbaijan          55
3 2018-01-02      Bahrain          20
4 2018-01-02        Egypt           3


In [27]:
regionalACLEDFiltered.head()

,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,latitude,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,tradingDay
0,EGY8633,2018-01-01,2018,1,Political violence,Violence against civilians,Attack,Unidentified Armed Group (Egypt),NaN,Political militia,...,30.0024,31.2019,1,Egypt Independent,National,Unidentified militants attacked a liquor shop ...,2,NaN,1618529456,2018-01-02
1,IRQ19369,2018-01-01,2018,1,Political violence,Battles,Armed clash,PKK: Kurdistan Workers Party,NaN,Rebel group,...,36.6543,44.5377,2,People's Defense Forces,Other,"HPG reported that on January 1, PKK attacked T...",0,NaN,1618558319,2018-01-02
2,IRN291,2018-01-01,2018,1,Political violence,Battles,Armed clash,Unidentified Armed Group (Iran),NaN,Political militia,...,32.6344,51.3668,1,AFP,International,A policeman was shot dead in Iran on Monday in...,1,NaN,1561470034,2018-01-02
3,YEM7879,2018-01-01,2018,1,Political violence,Explosions/Remote violence,Air/drone strike,Operation Restoring Hope,NaN,External/Other forces,...,16.9861,43.6798,2,Yemen News Agency SABA,National,Three airstrikes were Al-anad area of Sahar di...,0,NaN,1561471478,2018-01-02
4,IRN290,2018-01-01,2018,1,Demonstrations,Protests,Peaceful protest,Protesters (Iran),Labor Group (Iran),Protesters,...,38.0456,46.2005,1,Iranian Labour News Agency,National,Workers of Tabriz Industrial Machines company ...,0,NaN,1618556715,2018-01-02


In [28]:
uniqueTradingDays = dailyCountryCounts['tradingDay'].unique()
print(len(uniqueTradingDays))
print(uniqueTradingDays[:5])

1879
<DatetimeArray>
['2018-01-02 00:00:00', '2018-01-03 00:00:00', '2018-01-04 00:00:00',
 '2018-01-05 00:00:00', '2018-01-08 00:00:00']
Length: 5, dtype: datetime64[us]


In [29]:
allTradingWindows = []

for day in uniqueTradingDays: 
    window = getEventWindow(day, marketReturns, before=1, after=3)
    if window is not None: 
        window['tradingDay'] = day
        allTradingWindows.append(window)

marketWindows = pd.concat(allTradingWindows)
print(marketWindows.shape)
print(marketWindows.head())

(9390, 12)
            Close_EEM   Close_GLD  Close_ITA  Close_USO  Close_VIX  \
Date                                                                 
2018-01-02  39.776558  125.150002  86.677139  96.559998       9.77   
2018-01-03  40.157669  124.820000  86.805916  98.720001       9.15   
2018-01-04  40.356510  125.459999  87.413139  98.959999       9.22   
2018-01-05  40.704487  125.330002  88.199722  98.480003       9.22   
2018-01-08  40.704487  125.309998  88.733322  99.040001       9.52   

            Return_EEM  Return_GLD  Return_ITA  Return_USO  Return_VIX  \
Date                                                                     
2018-01-02         NaN         NaN         NaN         NaN         NaN   
2018-01-03    0.009581   -0.002637    0.001486    0.022370   -0.063460   
2018-01-04    0.004952    0.005127    0.006995    0.002431    0.007650   
2018-01-05    0.008623   -0.001036    0.008998   -0.004850    0.000000   
2018-01-08    0.000000   -0.000160    0.006050    0.00

In [30]:
alignedCountryData = dailyCountryCounts.merge(marketWindows, on='tradingDay', how='inner')
print(alignedCountryData.shape)
print(alignedCountryData.head())

(367530, 14)
  tradingDay      country  eventCount  Close_EEM   Close_GLD  Close_ITA  \
0 2018-01-03  Afghanistan          26  39.776558  125.150002  86.677139   
1 2018-01-03  Afghanistan          26  40.157669  124.820000  86.805916   
2 2018-01-03  Afghanistan          26  40.356510  125.459999  87.413139   
3 2018-01-03  Afghanistan          26  40.704487  125.330002  88.199722   
4 2018-01-03  Afghanistan          26  40.704487  125.309998  88.733322   

   Close_USO  Close_VIX  Return_EEM  Return_GLD  Return_ITA  Return_USO  \
0  96.559998       9.77         NaN         NaN         NaN         NaN   
1  98.720001       9.15    0.009581   -0.002637    0.001486    0.022370   
2  98.959999       9.22    0.004952    0.005127    0.006995    0.002431   
3  98.480003       9.22    0.008623   -0.001036    0.008998   -0.004850   
4  99.040001       9.52    0.000000   -0.000160    0.006050    0.005686   

   Return_VIX  offset  
0         NaN      -1  
1   -0.063460       0  
2    0.007650

In [31]:
closeCols = ['Close_EEM', 'Close_GLD', 'Close_ITA', 'Close_USO', 'Close_VIX']

alignedCountryData = alignedCountryData.drop(columns=closeCols)

print(alignedCountryData.shape)
print(alignedCountryData.head())

(367530, 9)
  tradingDay      country  eventCount  Return_EEM  Return_GLD  Return_ITA  \
0 2018-01-03  Afghanistan          26         NaN         NaN         NaN   
1 2018-01-03  Afghanistan          26    0.009581   -0.002637    0.001486   
2 2018-01-03  Afghanistan          26    0.004952    0.005127    0.006995   
3 2018-01-03  Afghanistan          26    0.008623   -0.001036    0.008998   
4 2018-01-03  Afghanistan          26    0.000000   -0.000160    0.006050   

   Return_USO  Return_VIX  offset  
0         NaN         NaN      -1  
1    0.022370   -0.063460       0  
2    0.002431    0.007650       1  
3   -0.004850    0.000000       2  
4    0.005686    0.032538       3  


In [32]:
alignedCountryData.to_csv('../Data/Processed/alignedCountryData.csv', index=False)

In [33]:
print(alignedCountryData.columns.tolist())

['tradingDay', 'country', 'eventCount', 'Return_EEM', 'Return_GLD', 'Return_ITA', 'Return_USO', 'Return_VIX', 'offset']


In [34]:
countryRegionMap = regionalACLEDFiltered[['country', 'region']].drop_duplicates()
print(countryRegionMap.shape)
print(countryRegionMap.head())

(69, 2)
   country           region
0    Egypt  Northern Africa
1     Iraq      Middle East
2     Iran      Middle East
3    Yemen      Middle East
5  Ukraine           Europe


In [35]:
alignedCountryData = alignedCountryData.merge(countryRegionMap, on='country', how='left')

print(alignedCountryData.shape)
print(alignedCountryData.head())
print(alignedCountryData['region'].isna().sum())

(367530, 10)
  tradingDay      country  eventCount  Return_EEM  Return_GLD  Return_ITA  \
0 2018-01-03  Afghanistan          26         NaN         NaN         NaN   
1 2018-01-03  Afghanistan          26    0.009581   -0.002637    0.001486   
2 2018-01-03  Afghanistan          26    0.004952    0.005127    0.006995   
3 2018-01-03  Afghanistan          26    0.008623   -0.001036    0.008998   
4 2018-01-03  Afghanistan          26    0.000000   -0.000160    0.006050   

   Return_USO  Return_VIX  offset                     region  
0         NaN         NaN      -1  Caucasus and Central Asia  
1    0.022370   -0.063460       0  Caucasus and Central Asia  
2    0.002431    0.007650       1  Caucasus and Central Asia  
3   -0.004850    0.000000       2  Caucasus and Central Asia  
4    0.005686    0.032538       3  Caucasus and Central Asia  
0


In [36]:
alignedCountryData.head()

,tradingDay,country,eventCount,Return_EEM,Return_GLD,Return_ITA,Return_USO,Return_VIX,offset,region
0,2018-01-03,Afghanistan,26,NaN,NaN,NaN,NaN,NaN,-1,Caucasus and Central Asia
1,2018-01-03,Afghanistan,26,0.009581,-0.002637,0.001486,0.022370,-0.063460,0,Caucasus and Central Asia
2,2018-01-03,Afghanistan,26,0.004952,0.005127,0.006995,0.002431,0.007650,1,Caucasus and Central Asia
3,2018-01-03,Afghanistan,26,0.008623,-0.001036,0.008998,-0.004850,0.000000,2,Caucasus and Central Asia
4,2018-01-03,Afghanistan,26,0.000000,-0.000160,0.006050,0.005686,0.032538,3,Caucasus and Central Asia


In [37]:
alignedCountryData.to_csv('../Data/Processed/alignedDataCountry.csv', index=False)

In [38]:
alignedCountryData.head()

,tradingDay,country,eventCount,Return_EEM,Return_GLD,Return_ITA,Return_USO,Return_VIX,offset,region
0,2018-01-03,Afghanistan,26,NaN,NaN,NaN,NaN,NaN,-1,Caucasus and Central Asia
1,2018-01-03,Afghanistan,26,0.009581,-0.002637,0.001486,0.022370,-0.063460,0,Caucasus and Central Asia
2,2018-01-03,Afghanistan,26,0.004952,0.005127,0.006995,0.002431,0.007650,1,Caucasus and Central Asia
3,2018-01-03,Afghanistan,26,0.008623,-0.001036,0.008998,-0.004850,0.000000,2,Caucasus and Central Asia
4,2018-01-03,Afghanistan,26,0.000000,-0.000160,0.006050,0.005686,0.032538,3,Caucasus and Central Asia


In [39]:
from Pipeline.build_country_comparisons import buildCountryComparisons

countryComparisons = buildCountryComparisons(alignedCountryData)

print(countryComparisons.shape)
print(countryComparisons.head())

(232525, 6)
                    week                     region     countryI    countryJ  \
0  2018-01-01/2018-01-07  Caucasus and Central Asia  Afghanistan     Armenia   
1  2018-01-01/2018-01-07  Caucasus and Central Asia  Afghanistan  Azerbaijan   
2  2018-01-01/2018-01-07  Caucasus and Central Asia  Afghanistan     Georgia   
3  2018-01-01/2018-01-07  Caucasus and Central Asia  Afghanistan  Kazakhstan   
4  2018-01-01/2018-01-07  Caucasus and Central Asia  Afghanistan  Kyrgyzstan   

   winsI  total  
0     97    104  
1     97    190  
2     97    105  
3     97    100  
4     97     98  


In [40]:
countryToRegion = alignedCountryData[['country', 'region']].drop_duplicates()

print(countryToRegion.head())

        country                     region
0   Afghanistan  Caucasus and Central Asia
5       Algeria            Northern Africa
10      Armenia  Caucasus and Central Asia
15   Azerbaijan  Caucasus and Central Asia
20      Bahrain                Middle East


In [41]:
regionNameMap = {
    'Caucasus and Central Asia' : 0, 
    'Europe' : 1, 
    'Middle East' : 2, 
    'Northern Africa' : 3
}

countries = sorted(countryToRegion['country'].unique())
countryToRegionMap = dict()

for country in countries: 
    regionName = countryToRegion[countryToRegion['country'] == country]['region'].iloc[0]
    countryToRegionMap[country] = regionNameMap[regionName]

print(len(countries))
print(list(countryToRegionMap.items())[:5])

69
[('Afghanistan', 0), ('Albania', 1), ('Algeria', 3), ('Armenia', 0), ('Austria', 1)]


In [42]:
countryIndexMap = {country: i for i, country in enumerate(countries)}

winsI = countryComparisons['winsI'].to_numpy()
totals = countryComparisons['total'].to_numpy()

indexI = countryComparisons['countryI'].map(countryIndexMap).to_numpy()
indexJ = countryComparisons['countryJ'].map(countryIndexMap).to_numpy()

regionOfCountry = np.array([countryToRegionMap[country] for country in countries])

nCountries = len(countries)
nRegions = len(regionNameMap)

print(indexI[:10])
print(indexJ[:10])
print(regionOfCountry[:10])

[0 0 0 0 0 0 3 3 3 3]
[ 3  5 19 30 33 67  5 19 30 33]
[0 1 3 0 1 0 2 1 1 1]


In [46]:
testComparisons = countryComparisons[countryComparisons['region'] == 'Caucasus and Central Asia'].copy()

testCountries = sorted(set(testComparisons['countryI']) | set(testComparisons['countryJ']))
testCountryIndexMap = {country: i for i, country in enumerate(testCountries)}

testWinsI = testComparisons['winsI'].to_numpy()
testTotals = testComparisons['total'].to_numpy()
testIdxI = testComparisons['countryI'].map(testCountryIndexMap).to_numpy()
testIdxJ = testComparisons['countryJ'].map(testCountryIndexMap).to_numpy()

nTestCountries = len(testCountries)

print(nTestCountries)
print(testComparisons.shape)

9
(8523, 6)


In [55]:
with pm.Model() as testModel4:
    countryBetaTest = pm.ZeroSumNormal('countryBetaTest', sigma=1.0, shape=nTestCountries)

    strengthITest = countryBetaTest[testIdxI]
    strengthJTest = countryBetaTest[testIdxJ]

    logOddsTest = strengthITest - strengthJTest
    probabilityTest = pm.math.sigmoid(logOddsTest)

    likelihoodTest = pm.Binomial('likelihoodTest', n=testTotals, p=probabilityTest, observed=testWinsI)

with testModel4:
    testTrace4 = pm.sample(500, tune=500, target_accept=0.9, return_inferencedata=True, chains=2, cores=2, progressbar='combined')

print(az.summary(testTrace4))

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [countryBetaTest]


c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 15 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


                     mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  \
countryBetaTest[0]  2.692  0.009   2.677    2.708      0.000    0.000   
countryBetaTest[1]  0.654  0.009   0.637    0.672      0.000    0.000   
countryBetaTest[2]  1.643  0.009   1.628    1.659      0.000    0.000   
countryBetaTest[3]  0.313  0.009   0.295    0.330      0.000    0.000   
countryBetaTest[4]  0.371  0.010   0.351    0.389      0.001    0.000   
countryBetaTest[5] -0.477  0.011  -0.496   -0.456      0.000    0.000   
countryBetaTest[6] -1.895  0.028  -1.949   -1.844      0.001    0.001   
countryBetaTest[7] -2.318  0.049  -2.410   -2.227      0.002    0.002   
countryBetaTest[8] -0.982  0.014  -1.008   -0.956      0.000    0.000   

                    ess_bulk  ess_tail  r_hat  
countryBetaTest[0]     370.0     620.0   1.00  
countryBetaTest[1]     459.0     601.0   1.00  
countryBetaTest[2]     384.0     611.0   1.01  
countryBetaTest[3]     458.0     613.0   1.00  
countryBetaTest[4]     384.0 

In [58]:
regionTraces = dict()
regionCountryLists = dict()

for regionName in countryComparisons['region'].unique(): 
    regionData = countryComparisons[countryComparisons['region'] == regionName]

    regionCountries = sorted(set(regionData['countryI']) | set(regionData['countryJ']))
    regionIndexMap = {country: i for i, country in enumerate(regionCountries)}

    rWinsI = regionData['winsI'].to_numpy()
    rTotals = regionData['total'].to_numpy()
    rIndexI = regionData['countryI'].map(regionIndexMap).to_numpy()
    rIndexJ = regionData['countryJ'].map(regionIndexMap).to_numpy()

    with pm.Model() as regionModel: 
        beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=len(regionCountries))
        probability = pm.math.sigmoid(beta[rIndexI] - beta[rIndexJ])
        observed = pm.Binomial('observed', n=rTotals, p=probability, observed=rWinsI)

        trace = pm.sample(500, tune=500, target_accept=0.9, return_inferencedata=True, chains=2, cores=2)

    regionTraces[regionName] = trace
    regionCountryLists[regionName] = regionCountries

    print(f'{regionName}: done, {len(regionCountries)} countries')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta]


c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 13 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Caucasus and Central Asia: done, 9 countries


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta]


...

Middle East: done, 13 countries


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta]


c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 11 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


Northern Africa: done, 6 countries


In [59]:
for regionName, trace in regionTraces.items():
    print(f"--- {regionName} ---")
    print(az.summary(trace))
    print()

--- Caucasus and Central Asia ---
          mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
beta[0]  2.692  0.008   2.677    2.708      0.000    0.000     523.0   
beta[1]  0.654  0.009   0.636    0.671      0.000    0.000     599.0   
beta[2]  1.643  0.008   1.626    1.657      0.000    0.000     657.0   
beta[3]  0.314  0.009   0.298    0.333      0.000    0.000     680.0   
beta[4]  0.371  0.010   0.352    0.388      0.000    0.000     793.0   
beta[5] -0.477  0.011  -0.497   -0.457      0.000    0.000     823.0   
beta[6] -1.894  0.027  -1.946   -1.844      0.001    0.001     982.0   
beta[7] -2.321  0.046  -2.405   -2.230      0.002    0.001     590.0   
beta[8] -0.982  0.014  -1.007   -0.956      0.000    0.000    2446.0   

         ess_tail  r_hat  
beta[0]     824.0    1.0  
beta[1]     680.0    1.0  
beta[2]     736.0    1.0  
beta[3]     574.0    1.0  
beta[4]     767.0    1.0  
beta[5]     690.0    1.0  
beta[6]     601.0    1.0  
beta[7]     712.0    1.0  
be

In [60]:
regionUncertainty = {}

for regionName, trace in regionTraces.items():
    regionCountries = regionCountryLists[regionName]
    results = computeUncertainty(trace, regionCountries)
    regionUncertainty[regionName] = results

    print(f"--- {regionName} ---")
    print("Entropy:", results['entropy'])
    print("Flip probabilities:", results['flipProbabilities'])
    print("HDI overlaps:", results['hdiOverlaps'])
    print()

--- Caucasus and Central Asia ---
Entropy: 0.0
Flip probabilities: {'Afghanistan over Armenia': np.float64(1.0), 'Afghanistan over Azerbaijan': np.float64(1.0), 'Afghanistan over Georgia': np.float64(1.0), 'Afghanistan over Kazakhstan': np.float64(1.0), 'Afghanistan over Kyrgyzstan': np.float64(1.0), 'Afghanistan over Tajikistan': np.float64(1.0), 'Afghanistan over Turkmenistan': np.float64(1.0), 'Afghanistan over Uzbekistan': np.float64(1.0), 'Armenia over Azerbaijan': np.float64(0.0), 'Armenia over Georgia': np.float64(1.0), 'Armenia over Kazakhstan': np.float64(1.0), 'Armenia over Kyrgyzstan': np.float64(1.0), 'Armenia over Tajikistan': np.float64(1.0), 'Armenia over Turkmenistan': np.float64(1.0), 'Armenia over Uzbekistan': np.float64(1.0), 'Azerbaijan over Georgia': np.float64(1.0), 'Azerbaijan over Kazakhstan': np.float64(1.0), 'Azerbaijan over Kyrgyzstan': np.float64(1.0), 'Azerbaijan over Tajikistan': np.float64(1.0), 'Azerbaijan over Turkmenistan': np.float64(1.0), 'Azerbaijan